### Running agents
[Documentation](https://openai.github.io/openai-agents-python/running_agents/)

You can run agents via the Runner class. You have 3 options:

- **Runner.run()**, which runs async and returns a RunResult.
- **Runner.run_sync()**, which is a sync method and just runs .run() under the hood.
- **Runner.run_streamed()**, which runs async and returns a RunResultStreaming. It calls the LLM in streaming mode, and streams those events to you as they are received.


In [1]:
# Imports
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled)

In [2]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [35]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=external_client
)

### Running Agents Asynchronously

In [ ]:

agent = Agent(name="Assistant", instructions="You are a helpful assistant", model=llm_model)

result = await Runner.run(agent, "Write a haiku about recursion in programming.")
print(result.final_output)
# Code within the code,
# Functions calling themselves,
# Infinite loop's dance


Function calls itself,
With a smaller problem now,
Stack unwinds with grace.


### Running Agents Synchronously
```result = Runner.run_sync(agent2, "Write a haiku about recursion in programming.")```

Run this code in ```(.py)``` file

In [ ]:
# install nest_asyncio if you haven't already
agent2 = Agent(name="Assistant", instructions="You are a helpful assistant", model=llm_model)

result = Runner.run_sync(agent2, "Write a haiku about recursion in programming.")
print(result.final_output)
# Code within the code,
# Functions calling themselves,
# Infinite loop's dance

### Running Agents in Stream Mode

```result = Runner.run_streamed(agent, "Summarize recursion in one sentence.")```

##### In the OpenAI Agents SDK, stream_events() yields only three event types:

- **raw_response_event**: low-level model events, including the text deltas
- **run_item_stream_event**: higher-level items (messages, tool calls, and so on)
- **agent_updated_stream_event**: agent handoffs

```    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):```

##### Common `event.data` Types in Streaming (Responses API)
When you stream a simple text answer, `event.data` (inside a `raw_response_event`) usually arrives in this order:

| Order | Event Class | Meaning |
|-------|-------------|---------|
| 1 | `ResponseCreatedEvent` | The response has started |
| 2 | `ResponseInProgressEvent` | The model is working |
| 3 | `ResponseOutputItemAddedEvent` | A new output item (e.g., a message) begins |
| 4 | `ResponseContentPartAddedEvent` | A part of the content begins |
| 5 | **`ResponseTextDeltaEvent`** | **A small chunk of text (the one we print)** |
| 6 | `ResponseTextDoneEvent` | The text is finished |
| 7 | `ResponseContentPartDoneEvent` | The content part is finished |
| 8 | `ResponseOutputItemDoneEvent` | The output item is finished |
| 9 | `ResponseCompletedEvent` | The whole response is complete |

<br>

> **Note:** `ResponseTextDeltaEvent` is fired many times, once for each small piece of text. The other events usually fire once each.

In [38]:
from openai.types.responses import ResponseTextDeltaEvent
agent = Agent(name="Assistant", instructions="Be concise.", model=llm_model)
result = Runner.run_streamed(agent, "Summarize recursion in more details.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Recursion is a programming technique where a function calls itself to solve a problem by breaking it down into smaller, self-similar sub-problems.

### The Two Core Components
1.  **Base Case:** The termination condition that stops the recursion. Without it, the function enters an infinite loop, resulting in a stack overflow.
2.  **Recursive Step:** The part of the function that calls itself with a modified argument, moving the state closer to the base case.

### How It Works (The Call Stack)
Each time a function calls itself, a new **stack frame** is pushed onto the call stack. This frame holds the function's local variables and return address. Once the base case is reached, the results "bubble up" as the stack frames are popped off one by one, completing the original call.

### Pros and Cons
*   **Pros:** Leads to elegant, clean code for complex problems (e.g., tree traversals, quicksort, graph algorithms).
*   **Cons:** Higher memory consumption due to stack overhead; slower than it